In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, export_text

# ------------------ Step 1: Load Titanic Dataset ------------------
df = pd.read_csv('titanic.csv')
print("Dataset Loaded:")
print(df)
# Select relevant features and drop missing values
df =df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'Survived']].dropna()

# Convert target to 'yes'/'no' format for classification
df['Class'] = df['Survived'].apply(lambda x: 'yes' if x == 1 else 'no')
df.drop(columns='Survived', inplace=True)

# Convert categorical columns to dummy/one-hot encoding
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

print("Sample data:")
print(df.head())

# ------------------ Step 2: CN2-Like Rule Induction ------------------
X = df.drop(columns='Class')
y = df['Class']

tree = DecisionTreeClassifier(criterion='entropy', max_depth=4)
tree.fit(X, y)

print("\n[CN2-like] Rules learned using Decision Tree:")
rules = export_text(tree, feature_names=list(X.columns))
print(rules)

# ------------------ Step 3: FOIL-Like Rule Induction ------------------
def foil_like(dataframe, target_class='yes'):
    positives = dataframe[dataframe['Class'] == target_class]
    negatives = dataframe[dataframe['Class'] != target_class]

    final_rules = []

    for _, pos in positives.iterrows():
        rule = []
        for attr in dataframe.columns[:-1]:  # Exclude 'Class'
            val = pos[attr]
            if isinstance(val, (int, float)):
                rule.append(f"{attr} <= {round(val, 2)}")
            else:
                rule.append(f"{attr} = {val}")

        def rule_matches(row):
            for cond in rule:
                if '<=' in cond:
                    attr, value = cond.split('<=')
                    attr = attr.strip()
                    value = float(value.strip())
                    if float(row[attr]) > value:
                        return False
                elif '=' in cond:
                    attr, value = cond.split('=')
                    attr = attr.strip()
                    value = value.strip()
                    if str(row[attr]) != value:
                        return False
            return True

        match_neg = any(rule_matches(row) for _, row in negatives.iterrows())
        if not match_neg:
            final_rules.append(" AND ".join(rule))

    return final_rules[:5]  # Return only top 5 general rules

# Run FOIL-like learner
foil_rules = foil_like(df, target_class='yes')

print("\n[FOIL-like] Rules for Class = yes:")
for rule in foil_rules:
    print(f"IF {rule} THEN Class = yes")


Dataset Loaded:
     PassengerId  Survived  pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     sex   age  sibsp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ..